In [1]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[06/23/26 12:44:29] INFO     Using                                                                  ]8;id=320006;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=589304;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\                
                             project\rich_logging.yml' as logging configuration.                                   

[06/23/26 12:44:30] WARNING  c:\Users\User\miniconda3\envs\unis\Lib\site-packages\requests\__init__ ]8;id=721679;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=185290;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             .py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                        
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[06/23/26 12:44:31] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=457469;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=761212;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import unis_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import unis_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Cargar datos

In [3]:
df  = catalog.load('unis_estados_calac_survival')

[06/23/26 12:44:37] INFO     Loading data from unis_estados_calac_survival (ParquetDataset)... ]8;id=407481;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=331123;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [4]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

In [ ]:
# 2. Filtrar valores nulos en periodo_inicial
df = df[df['periodo_inicial'].notna()]

In [5]:
# --- Preparación del Modelo de Cox ---
# Seleccionamos las columnas necesarias para el modelo y la identificación posterior
columnas_interes = ['identificacion', 'periodo_inicial', 'nivel', 'programa', 'month', 'di']
df_modelo = df[columnas_interes].copy()

In [ ]:
# En Python (lifelines), el modelo requiere variables numéricas (dummies)
# Para simular relevel(ref = "marketing digital"), usamos drop_first o creamos dummies manualmente.
# Como solo usas 'programa' en la fórmula, creamos sus dummies:
df_dummies = pd.get_dummies(df_modelo['programa'], prefix='prog')

In [9]:

# Definimos la referencia eliminando la columna de 'marketing digital'
col_referencia = 'prog_marketing digital'
if col_referencia in df_dummies.columns:
    df_dummies = df_dummies.drop(columns=[col_referencia])

In [11]:
# Concatenamos las dummies al dataframe de trabajo
df_cox = pd.concat([df_modelo[['month', 'di']], df_dummies], axis=1)

In [13]:
# 3. Ajustar el modelo de Cox
best_model = CoxPHFitter()
best_model.fit(df_cox, duration_col='month', event_col='di')

<lifelines.CoxPHFitter: fitted with 1826 total observations, 1215 right-censored observations>

In [15]:
# 4. Obtener las curvas de supervivencia individualizadas
# 'predict_survival_function' devuelve un DataFrame donde:
# Índices (filas) = Tiempos en meses
# Columnas = Índices de los estudiante
surv_matrix = best_model.predict_survival_function(df_cox)

In [25]:
# 5. Extraer probabilidades para los meses_evaluar (1 a 16)
meses_evaluar = list(range(1, 17))

In [26]:
# Reindexamos para asegurar que tenemos exactamente los meses del 1 al 16.
# ffill() emula el 'extend = TRUE' de R (mantiene el último valor conocido si el mes exacto no tuvo eventos)
# bfill() asegura que si el mes 1 no está explícito, herede el 100% (1.0) del mes 0
surv_matrix_eval = surv_matrix.reindex(meses_evaluar).ffill().bfill()


In [21]:
# --- Reconstrucción del dataset final en formato largo (Unnest) ---
# Reseteamos el índice de la matriz para tener 'mes' como columna y pivoteamos a formato largo
surv_matrix_long = surv_matrix_eval.reset_index().rename(columns={'index': 'mes'})
surv_matrix_long = pd.melt(surv_matrix_long, id_vars=['mes'], var_name='estudiante_idx', value_name='p_sobrevivir')

In [ ]:
# Aseguramos que los tipos coincidan para hacer el cruce final
surv_matrix_long['estudiante_idx'] = surv_matrix_long['estudiante_idx'].astype(int)

# 6. Unir con las variables llave originales de df
df_probabilidades = df_modelo.reset_index(drop=True)
df_probabilidades['estudiante_idx'] = df_probabilidades.index

# Combinar todo para obtener el formato largo final por estudiante y mes
df_probabilidades = pd.merge(surv_matrix_long, df_probabilidades, on='estudiante_idx')

# Reordenar columnas para que quede idéntico a tu output de R
df_probabilidades = df_probabilidades[['identificacion', 'periodo_inicial', 'nivel', 'programa', 'month', 'di', 'p_sobrevivir', 'mes']]

# Visualizar el resultado final
print(df_probabilidades.head())

   identificacion  periodo_inicial     nivel                 programa  month  \
0    1.622559e+12           9243.0  maestria  neuropsicologia clinica      1   
1    1.622559e+12           9243.0  maestria  neuropsicologia clinica      1   
2    1.622559e+12           9243.0  maestria  neuropsicologia clinica      1   
3    1.622559e+12           9243.0  maestria  neuropsicologia clinica      1   
4    1.622559e+12           9243.0  maestria  neuropsicologia clinica      1   

   di  p_sobrevivir  mes  
0   1      0.915261    1  
1   1      0.873134    2  
2   1      0.823446    3  
3   1      0.779711    4  
4   1      0.760697    5  


In [29]:
df_probabilidades['programa'].unique()


array(['neuropsicologia clinica', 'docencia universitaria',
       'marketing digital', 'administracion financiera', 'talento humano',
       'direccion financiera', 'ciberseguridad', 'gestion talento humano'],
      dtype=object)

In [30]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. LISTA COMPLETA DE PROGRAMAS DEL MODELO ORIGINAL
# ==============================================================================
programas_originales = [
    'neuropsicologia clinica', 'docencia universitaria', 'marketing digital', 
    'administracion financiera', 'talento humano', 'direccion financiera', 
    'ciberseguridad', 'gestion talento humano'
]

# Definimos los nuevos ingresos operacionales (solo te interesan estos dos para proyectar)
nuevos_estudiantes = pd.DataFrame({
    'programa': ['marketing digital', 'administracion financiera'],
    'n_inicial': [150, 120]
})

# ==============================================================================
# 2. CONSTRUIR LA MATRIZ DE PREDICCIÓN CON TODAS LAS DUMMIES ORIGINALES
# ==============================================================================
# Creamos un DataFrame vacío que tendrá una fila por cada programa que queremos proyectar
# y una columna por cada variable dummy que el modelo 'best_model' espera recibir.
programas_a_proyectar = nuevos_estudiantes['programa'].tolist()

# Construimos la estructura idéntica de dummies que vio el modelo al entrenarse
columnas_dummies = [f'prog_{p}' for p in programas_originales if p != 'marketing digital']
df_pred_dummies = pd.DataFrame(0, index=range(len(programas_a_proyectar)), columns=columnas_dummies)

# Activamos el "1" en la dummy correspondiente para cada programa a proyectar
for i, prog in enumerate(programas_a_proyectar):
    col_name = f'prog_{prog}'
    if col_name in df_pred_dummies.columns:
        df_pred_dummies.loc[i, col_name] = 1
    # Si el programa es 'marketing digital', todas sus dummies se quedan en 0 (es la referencia)

# ==============================================================================
# 3. OBTENER LAS PREDICCIONES DEL MODELO DE COX
# ==============================================================================
# 'best_model' ahora recibe la estructura perfecta con todas las variables del modelo original
pred_cox = best_model.predict_survival_function(df_pred_dummies)

# Definimos el horizonte de meses a evaluar (del mes 1 al máximo del set original)
max_meses = int(df['month'].max())
meses_evaluar = list(range(1, max_meses + 1))

# Reindexamos y aplicamos forward-fill / backward-fill para construir la escalera continua
tabla_superv_base = pred_cox.reindex(meses_evaluar).ffill().bfill()

# Pasamos a formato largo (melt)
tabla_superv_base = tabla_superv_base.reset_index().rename(columns={'index': 'mes'})
tabla_superv_base = pd.melt(tabla_superv_base, id_vars=['mes'], var_name='programa_idx', value_name='p_sobrevivir')

# Mapeamos los índices numéricos de la predicción a los nombres de los programas reales
tabla_superv_base['programa'] = tabla_superv_base['programa_idx'].map(pd.Series(programas_a_proyectar))

# ==============================================================================
# 4. CALCULAR LA TABLA DE VIDA PROYECTADA (Totales Esperados)
# ==============================================================================
tabla_vida_nuevos = pd.merge(nuevos_estudiantes, tabla_superv_base, on='programa')

# Cálculos de la población esperada
tabla_vida_nuevos['estudiantes_esperados_activos'] = tabla_vida_nuevos['n_inicial'] * tabla_vida_nuevos['p_sobrevivir']
tabla_vida_nuevos['estudiantes_esperados_redond'] = tabla_vida_nuevos['estudiantes_esperados_activos'].round().astype(int)
tabla_vida_nuevos['desertores_acumulados_esperados'] = tabla_vida_nuevos['n_inicial'] - tabla_vida_nuevos['estudiantes_esperados_redond']

# Seleccionar y ordenar las columnas finales
tabla_vida_nuevos = tabla_vida_nuevos[[
    'programa', 'mes', 'n_inicial', 'p_sobrevivir', 
    'estudiantes_esperados_redond', 'desertores_acumulados_esperados'
]].sort_values(by=['programa', 'mes']).reset_index(drop=True)

print(tabla_vida_nuevos.head(20))

                     programa  mes  n_inicial  p_sobrevivir  \
0   administracion financiera    1        120      0.884415   
1   administracion financiera    2        120      0.828454   
2   administracion financiera    3        120      0.763784   
3   administracion financiera    4        120      0.708096   
4   administracion financiera    5        120      0.684256   
5   administracion financiera    6        120      0.620640   
6   administracion financiera    7        120      0.550274   
7   administracion financiera    8        120      0.541873   
8   administracion financiera    9        120      0.527011   
9   administracion financiera   10        120      0.502262   
10  administracion financiera   11        120      0.500703   
11  administracion financiera   12        120      0.496045   
12  administracion financiera   13        120      0.492947   
13  administracion financiera   14        120      0.488313   
14  administracion financiera   15        120      0.48

In [31]:
tabla_vida_nuevos

,programa,mes,n_inicial,p_sobrevivir,estudiantes_esperados_redond,desertores_acumulados_esperados
0,administracion financiera,1,120,0.884415,106,14
1,administracion financiera,2,120,0.828454,99,21
2,administracion financiera,3,120,0.763784,92,28
3,administracion financiera,4,120,0.708096,85,35
4,administracion financiera,5,120,0.684256,82,38
5,administracion financiera,6,120,0.620640,74,46
6,administracion financiera,7,120,0.550274,66,54
7,administracion financiera,8,120,0.541873,65,55
8,administracion financiera,9,120,0.527011,63,57
9,administracion financiera,10,120,0.502262,60,60
